# Exercises XP — Data understanding and import

Octopus Week 2 · Day 1

[Assignment](https://octopus.developers.institute/courses/collection/254/course/812/section/2143/chapter/4784)

Run cells in order. Source datasets are in `../data`; see the root README for setup and provenance.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis_utils.py').exists())
sys.path.insert(0, str(ROOT))
sns.set_theme(style='whitegrid')
def load(kind, filename=None, **kwargs):
    files = list((ROOT / 'data' / kind).rglob(filename or '*.csv'))
    if len(files) != 1:
        raise ValueError(f'Select an unambiguous data file: {files}')
    return pd.read_csv(files[0], **kwargs)
def savefig(name):
    target = ROOT / 'figures'
    target.mkdir(exist_ok=True)
    plt.tight_layout()
    plt.savefig(target / (name + '.png'), dpi=140, bbox_inches='tight')
    plt.show()


## Exercise 1 — What is data analysis?
Data analysis is a systematic process of collecting, validating, organizing and examining observations to answer a question. It combines domain knowledge, descriptive statistics, visualization and, where appropriate, statistical inference. A defensible analysis distinguishes a measured association from a causal explanation and records uncertainty and missing information.

It matters because raw records alone do not show whether a change is meaningful. Analysis makes decisions auditable, detects quality problems and helps allocate limited resources. A large sample still needs suitable definitions and a representative sampling process.

Three applications are: **retail**, where sales and margin analysis informs inventory and promotions; **health research**, where trials compare outcomes while accounting for uncertainty; and **weather forecasting**, where historical atmospheric observations support predictions and verification. ECMWF's operational AIFS deployment in 2025 illustrates the last application; see the sourced daily challenge report for detail.

## Exercise 2 — Load and inspect the three provided datasets
The assignment links GitHub mirrors, so no Kaggle credentials are needed. The mental-health file contains mixed-content rows; initial inspection does not silently treat all 108,553 rows as comparable observations.

In [2]:
sleep=load('sleep')
mental=load('mental_health',low_memory=False)
credit=load('credit','clean_dataset.csv')
for name,df in [('Sleep',sleep),('Mental health',mental),('Credit approvals',credit)]:
    print(name, 'shape:',df.shape)
    display(df.head())
    display(df.isna().sum().to_frame('missing'))
print('Sleep: annual average sleep hours, demographic groups and day types.')
print('Mental health: country/year health estimates; mixed-content rows require cleaning before comparison.')
print('Credit: applicant attributes and an approval outcome, using the supplied cleaned file.')

Sleep shape: (945, 9)


,index,Year,Period,Avg hrs per day sleeping,Standard Error,Type of Days,Age Group,Activity,Sex
0,0,2003,Annual,8.57,0.018,All days,15 years and over,Sleeping,Both
1,1,2004,Annual,8.55,0.026,All days,15 years and over,Sleeping,Both
2,2,2005,Annual,8.62,0.023,All days,15 years and over,Sleeping,Both
3,3,2006,Annual,8.63,0.024,All days,15 years and over,Sleeping,Both
4,4,2007,Annual,8.57,0.024,All days,15 years and over,Sleeping,Both


,missing
index,0
Year,0
Period,0
Avg hrs per day sleeping,0
Standard Error,0
Type of Days,0
Age Group,0
Activity,0
Sex,0


Mental health shape: (108553, 11)


,index,Entity,Code,Year,Schizophrenia (%),Bipolar disorder (%),Eating disorders (%),Anxiety disorders (%),Drug use disorders (%),Depression (%),Alcohol use disorders (%)
0,0,Afghanistan,AFG,1990,0.16056,0.697779,0.101855,4.828830,1.677082,4.071831,0.672404
1,1,Afghanistan,AFG,1991,0.160312,0.697961,0.099313,4.829740,1.684746,4.079531,0.671768
2,2,Afghanistan,AFG,1992,0.160135,0.698107,0.096692,4.831108,1.694334,4.088358,0.670644
3,3,Afghanistan,AFG,1993,0.160037,0.698257,0.094336,4.830864,1.705320,4.096190,0.669738
4,4,Afghanistan,AFG,1994,0.160022,0.698469,0.092439,4.829423,1.716069,4.099582,0.669260


,missing
index,0
Entity,0
Code,5412
Year,0
Schizophrenia (%),82678
Bipolar disorder (%),89147
Eating disorders (%),8317
Anxiety disorders (%),102085
Drug use disorders (%),102085
Depression (%),102085


Credit approvals shape: (690, 16)


,Gender,Age,Debt,Married,BankCustomer,Industry,Ethnicity,YearsEmployed,PriorDefault,Employed,CreditScore,DriversLicense,Citizen,ZipCode,Income,Approved
0,1,30.83,0.000,1,1,Industrials,White,1.25,1,1,1,0,ByBirth,202,0,1
1,0,58.67,4.460,1,1,Materials,Black,3.04,1,1,6,0,ByBirth,43,560,1
2,0,24.50,0.500,1,1,Materials,Black,1.50,1,0,0,0,ByBirth,280,824,1
3,1,27.83,1.540,1,1,Industrials,White,3.75,1,1,5,1,ByBirth,100,3,1
4,1,20.17,5.625,1,1,Industrials,White,1.71,1,0,0,0,ByOtherMeans,120,0,1


,missing
Gender,0
Age,0
Debt,0
Married,0
BankCustomer,0
Industry,0
Ethnicity,0
YearsEmployed,0
PriorDefault,0
Employed,0


Sleep: annual average sleep hours, demographic groups and day types.
Mental health: country/year health estimates; mixed-content rows require cleaning before comparison.
Credit: applicant attributes and an approval outcome, using the supplied cleaned file.


## Exercise 3 — Classify every column
Measurement role matters more than storage dtype: a numeric identifier or a coded category is qualitative. Year is a discrete time coordinate. The mental-health rate labels describe quantitative measures even when malformed rows force string storage.

In [3]:
qualitative={
 'Sleep':{'index','Period','Type of Days','Age Group','Activity','Sex'},
 'Mental health':{'index','Entity','Code'},
 'Credit approvals':{'Gender','Married','BankCustomer','Industry','Ethnicity','PriorDefault','Employed','DriversLicense','Citizen','ZipCode','Approved'}
}
for name,df in [('Sleep',sleep),('Mental health',mental),('Credit approvals',credit)]:
    rows=[]
    for col in df.columns:
        is_qual=col in qualitative[name]
        reason=('Identifier or category: arithmetic on the code is not meaningful.' if is_qual else 'Numeric amount, count, rate or time coordinate: differences have meaning.')
        rows.append({'column':col,'type':'qualitative' if is_qual else 'quantitative','reason':reason})
    print(name)
    display(pd.DataFrame(rows))

Sleep


,column,type,reason
0,index,qualitative,Identifier or category: arithmetic on the code...
1,Year,quantitative,"Numeric amount, count, rate or time coordinate..."
2,Period,qualitative,Identifier or category: arithmetic on the code...
3,Avg hrs per day sleeping,quantitative,"Numeric amount, count, rate or time coordinate..."
4,Standard Error,quantitative,"Numeric amount, count, rate or time coordinate..."
5,Type of Days,qualitative,Identifier or category: arithmetic on the code...
6,Age Group,qualitative,Identifier or category: arithmetic on the code...
7,Activity,qualitative,Identifier or category: arithmetic on the code...
8,Sex,qualitative,Identifier or category: arithmetic on the code...


Mental health


,column,type,reason
0,index,qualitative,Identifier or category: arithmetic on the code...
1,Entity,qualitative,Identifier or category: arithmetic on the code...
2,Code,qualitative,Identifier or category: arithmetic on the code...
3,Year,quantitative,"Numeric amount, count, rate or time coordinate..."
4,Schizophrenia (%),quantitative,"Numeric amount, count, rate or time coordinate..."
5,Bipolar disorder (%),quantitative,"Numeric amount, count, rate or time coordinate..."
6,Eating disorders (%),quantitative,"Numeric amount, count, rate or time coordinate..."
7,Anxiety disorders (%),quantitative,"Numeric amount, count, rate or time coordinate..."
8,Drug use disorders (%),quantitative,"Numeric amount, count, rate or time coordinate..."
9,Depression (%),quantitative,"Numeric amount, count, rate or time coordinate..."


Credit approvals


,column,type,reason
0,Gender,qualitative,Identifier or category: arithmetic on the code...
1,Age,quantitative,"Numeric amount, count, rate or time coordinate..."
2,Debt,quantitative,"Numeric amount, count, rate or time coordinate..."
3,Married,qualitative,Identifier or category: arithmetic on the code...
4,BankCustomer,qualitative,Identifier or category: arithmetic on the code...
5,Industry,qualitative,Identifier or category: arithmetic on the code...
6,Ethnicity,qualitative,Identifier or category: arithmetic on the code...
7,YearsEmployed,quantitative,"Numeric amount, count, rate or time coordinate..."
8,PriorDefault,qualitative,Identifier or category: arithmetic on the code...
9,Employed,qualitative,Identifier or category: arithmetic on the code...


## Exercise 4 — Iris feature classification

In [4]:
iris=load('iris')
display(iris.head())
display(pd.DataFrame([{'column':c,'type':'qualitative' if c in ['Id','Species'] else 'quantitative','reason':('Row identifier, not a measurement' if c=='Id' else 'Nominal species label' if c=='Species' else 'Continuous flower dimension measured in centimetres')} for c in iris.columns]))

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


,column,type,reason
0,Id,qualitative,"Row identifier, not a measurement"
1,SepalLengthCm,quantitative,Continuous flower dimension measured in centim...
2,SepalWidthCm,quantitative,Continuous flower dimension measured in centim...
3,PetalLengthCm,quantitative,Continuous flower dimension measured in centim...
4,PetalWidthCm,quantitative,Continuous flower dimension measured in centim...
5,Species,qualitative,Nominal species label


## Exercise 5 — Questions supported by the sleep data
Use **Year** and **Avg hrs per day sleeping** to study change over time, retaining a single sex/age/day-type stratum. Compare **Type of Days** within the same year and demographic stratum for weekday versus weekend differences. **Sex** and **Age Group** support group comparisons, but the “Both” and “15 years and over” aggregates overlap their component groups. They must not be pooled as independent people. **Standard Error** communicates uncertainty; this file contains survey estimates, not individual sleep records.

In [5]:
print('Observed labels:')
for c in ['Age Group','Sex','Type of Days']:
    print(c, sorted(sleep[c].dropna().unique()))
display(sleep[['Year','Avg hrs per day sleeping','Standard Error','Type of Days','Age Group','Sex']].head())

Observed labels:
Age Group ['15 to 24 years', '15 years and over', '25 to 34 years', '35 to 44 years', '45 to 54 years', '55 to 64 years', '65 years and over']
Sex ['Both', 'Men', 'Women']
Type of Days ['All days', 'Nonholiday weekdays', 'Weekend days and holidays']


,Year,Avg hrs per day sleeping,Standard Error,Type of Days,Age Group,Sex
0,2003,8.57,0.018,All days,15 years and over,Both
1,2004,8.55,0.026,All days,15 years and over,Both
2,2005,8.62,0.023,All days,15 years and over,Both
3,2006,8.63,0.024,All days,15 years and over,Both
4,2007,8.57,0.024,All days,15 years and over,Both


## Exercise 6 — Structured versus unstructured
| Source | Classification | Reason |
|---|---|---|
| Financial reports in Excel | Structured when organized as a table | Named columns and repeated records; narrative notes remain unstructured |
| Social photographs | Unstructured | Pixel content does not follow a business table schema; metadata may be structured |
| News articles | Unstructured text | Free narrative, even if dates and authors are structured metadata |
| Relational inventory database | Structured | Typed columns, keys and constraints |
| Recorded research interviews | Unstructured | Audio requires transcription and coding |

## Exercise 7 — Transform unstructured sources
| Source | Structured representation and rationale |
|---|---|
| Travel blog posts | One row per post: author, date, locations, topics and sentiment; extract metadata and named places, preserving text and source URL for review |
| Customer calls | Transcribe with timestamps and speaker turns; code issue, resolution and satisfaction into call/turn tables; review transcription errors and consent |
| Handwritten brainstorm | OCR plus human correction; record note ID, author if known, idea, theme and links; retain the scan to audit uncertain handwriting |
| Cooking video | Segment steps with start/end times; transcribe speech and annotate ingredient, amount, action and temperature; validate units against the video |

Extraction creates useful structure but does not guarantee accuracy; keep original sources and uncertainty flags.

## Exercise 8 — Import the provided Titanic train.csv

In [6]:
titanic=load('titanic')
display(titanic.head())
assert titanic.shape==(891,12)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Exercise 9 — Export to Excel and JSON

In [7]:
exports=ROOT/'day1'/'exports'
exports.mkdir(exist_ok=True)
sample=pd.DataFrame({'product':['Notebook','Pen','Folder'],'quantity':[3,10,2],'price':[4.5,1.2,2.0]})
sample.to_excel(exports/'products.xlsx',index=False)
sample.to_json(exports/'products.json',orient='records',indent=2)
pd.testing.assert_frame_equal(sample,pd.read_excel(exports/'products.xlsx'),check_dtype=False)
pd.testing.assert_frame_equal(sample,pd.read_json(exports/'products.json'),check_dtype=False)
display(sample)
print('Both exported files were read back and verified.')

,product,quantity,price
0,Notebook,3,4.5
1,Pen,10,1.2
2,Folder,2,2.0


Both exported files were read back and verified.


## Exercise 10 — Read JSON from the supplied URL
The supplied URL is a ZIP containing JSON. The download script retrieves that exact archive and extracts it; Pandas reads the cached payload. This avoids mistaking ZIP bytes for a JSON document.

In [8]:
posts=pd.read_json(next((ROOT/'data'/'posts').rglob('*.json')))
display(posts.head(5))
assert len(posts.head(5))==5

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...
